# E1 — La métrique affichée et la facture

**Exercice individuel noté, 5 % de la note du cours.** Séance 5. Notions testées : S1 à S4.
Terrain : la compétition de classe, dont vous avez le carnet de route depuis la séance 1.

> **Ce que ce n'est pas.** Ce n'est pas un exercice de code, et ce n'est pas un formulaire.
> Le livrable est une **mesure**, un **verdict**, et une **interface**.

## Comment ce gabarit est fait

Il ne contient **aucune réponse**, et **aucun emplacement vide à remplir**. Chaque question
donne un contexte, une consigne, les noms des objets à produire et le critère de réussite.
En dessous, une cellule ne porte **que les contrôles**.

Un contrôle vérifie la **forme** : un type, une plage, une dimension, une cohérence avec vos
autres réponses. Il ne compare jamais à une valeur attendue, et il **se tait** tant que la
réponse n'est pas écrite. Un cahier vierge n'affiche donc ni exception ni REVOIR.

**Ce que le gabarit vous donne** : la chaîne du cours, le modèle entraîné, les deux montants
de coût, et quatre objets que la séance 4 avait déjà mesurés — la courbe du coût, la formule
du seuil, le seuil appris, et le tableau des stratégies **non trié**. Ils sont là parce que
les recalculer ne vous apprendrait rien de plus que la séance 4.

**Ce que vous produisez** : seize objets nommés, l'interface `predict`, et un verdict de dix
lignes. Aucun n'est écrit dans un carnet.

**Volume.** Le créneau de la séance 5 ne suffit pas : comptez environ deux heures, dont la
moitié au temps personnel de l'étudiant. Le dépôt se fait à la clôture de la séance.

**Dépôt.** Ce carnet exécuté en `Save & Run All` (privé), le relevé `verifications.csv`, le
module `.py` qui expose `predict(df)`, et `declaration_E1.md` copié de `DECLARATION_modele.md`
— **dans le même commit**. Le feu de l'intelligence artificielle est 🟠 **orange** : l'IA est
autorisée pour écrire, refactorer, expliquer et déboguer, avec déclaration de ce qui a été
fait et de ce qui a été vérifié. Elle est interdite sur une chose : rendre un pipeline dont
vous ne pouvez expliquer aucune étape.


## Les outils du TD

Exécutez cette cellule, ne la modifiez pas. Elle pose le compteur de contrôles et les deux
fonctions qui les affichent.


In [1]:
# les outils du TD — exécutez cette cellule, ne la modifiez pas
# Le TD corrige la forme de vos réponses, jamais leur valeur : il ne vous donne aucune réponse.

try:
    from IPython.display import display
except ImportError:
    def display(x):
        print(x)

VERIFICATIONS = []


def controle(nom, quoi, indice='', des_que=None):
    """Contrôle de forme, posé à la fin d'une question. Il ne compare jamais à une valeur
       attendue : un type, une plage, une dimension, un compte. Tant que votre réponse n'est
       pas écrite, il se tait — et il le dit.

       `des_que` : la garde d'une réponse qui a une valeur par défaut. Tant que la garde est
       vide, le contrôle se tait."""
    if des_que is not None:
        try:
            if not des_que():
                print(f'  (pas encore écrit)  {nom}')
                return
        except NameError:
            print(f'  (pas encore écrit)  {nom}')
            return
    try:
        ok = bool(quoi())
    except NameError:
        print(f'  (pas encore écrit)  {nom}')
        return
    except Exception as e:
        print(f'  REVOIR    {nom}   ->   {type(e).__name__} : {e}')
        return
    VERIFICATIONS.append({'contrôle': nom, 'tient': ok})
    print(('  ok        ' if ok else '  REVOIR    ') + nom + ('' if ok else '   ->   ' + indice))


def bilan():
    """Le compte de vos contrôles : ce qui tient, et ce qu'il reste à revoir."""
    tenus = sum(1 for v in VERIFICATIONS if v['tient'])
    print(f"vos contrôles : {tenus} ok sur {len(VERIFICATIONS)}")
    reste = [v['contrôle'] for v in VERIFICATIONS if not v['tient']]
    if reste:
        print('  à revoir :', reste)


## Mise en place (fournie)

Ces cellules sont **extraites** des carnets des séances précédentes. Exécutez-les de haut en
bas, sans rien y modifier. Elles apportent les données, la table de travail, la référence, le
découpage, le modèle, la matrice de confusion et le coût.

Les `random_state=0` que vous verrez sont la graine du tirage : le même nombre redonne le même
découpage, donc la même mesure.

**Deux mots avant de commencer.** Le **positif** est l'**absence du patient** : c'est la classe
rare. Et **alerter**, c'est **annoncer une absence**.


### Les outils du cours


In [2]:
# les outils
# Aucun import en bloc au début : chaque module entre dans le carnet au moment de son premier
# usage. On voit ainsi à quoi il sert, au lieu de le deviner dans une liste.

# `display` : un tableau s'affiche proprement dans un carnet, et reste lisible ailleurs.
# C'est par `display` qu'on montre TOUTES les tables de ce carnet — jamais un print de texte.
try:
    from IPython.display import display
except ImportError:
    def display(x):
        print(x)


### Les données


In [3]:
import os
import pandas as pd
# où sont les fichiers — on CHERCHE `train.csv`, on ne suppose pas un chemin
# (Kaggle a déjà déplacé ses points de montage : un chemin recopié se périme, une recherche non)
def _trouver_dossier(racines):
    """Première racine contenant train.csv — en surface, puis en profondeur."""
    for racine in racines:
        if not os.path.isdir(racine):
            continue
        if os.path.exists(os.path.join(racine, "train.csv")):
            return racine
        for chemin, _, fichiers in os.walk(racine):
            if "train.csv" in fichiers:
                return chemin
    return None

DOSSIER = _trouver_dossier(["/kaggle/input", "sortie"])
assert DOSSIER, "train.csv introuvable : les données de la compétition sont-elles montées ?"

print("le dossier des données trouvé :", DOSSIER)


le dossier des données trouvé : /kaggle/input/competitions/rappels-et-sur-réservation-des-créneaux


In [4]:
train = pd.read_csv(os.path.join(DOSSIER, "train.csv"), low_memory=False)
test = pd.read_csv(os.path.join(DOSSIER, "test.csv"), low_memory=False)

ID, CIBLE = "ligne_id", "patient_absent"

print("les deux tableaux \n")
print("train :", train.shape, "(lignes, colonnes)")
print("test  :", test.shape, "(lignes, colonnes)")

print("les cinq premières lignes \n")
display(train.head())


les deux tableaux 

train : (149894, 39) (lignes, colonnes)
test  : (63721, 38) (lignes, colonnes)
les cinq premières lignes 



,dossier_id,ligne_id,patient_id,clinique,district,age,sexe,quartier,assurance,hypertension,...,indice_risque_externe,indice_meteo_jour,zone_clinique,tournee_jour,cout_acte,age_mois,score_territorial,canal_prise,taux_absence_passe,patient_absent
0,DOS-0127912,127913,P070425,C02,D3,53,F,Q04,privée,0,...,86.64,35,urbaine,T00000,10.4,636,-1.1621,guichet,0.0,1
1,DOS-0002883,2884,P109083,C36,D6,38,F,Q01,mutuelle,0,...,40.91,37,urbaine,T00002,5.3,456,-0.0963,guichet,0.0,0
2,DOS-0098561,98562,P036240,C43,D5,35,M,Q52,publique,1,...,28.34,82,urbaine,T00004,13.5,420,0.7951,en ligne,0.0,0
3,DOS-0133156,133157,P033484,C10,D2,36,M,Q07,aucune,0,...,55.91,76,urbaine,T00005,20.4,432,0.3788,guichet,0.0,0
4,DOS-0055744,55745,P035934,C28,D6,32,M,Q02,aucune,0,...,29.68,65,urbaine,T00006,8.8,384,1.2562,téléphone,0.0,0


### La table de travail


In [5]:
A_ECARTER = [ID, "dossier_id", "patient_id", "duree_consultation", "constantes_relevees",
             "motif_absence", "date_prise", "date_rdv", "taille_cm", "date_naissance",
             "canal_prise"]

def matrice(cols):
    """La table de travail : colonnes écartées retirées, catégorielles déclarées."""
    X = cols.drop(columns=[c for c in A_ECARTER + [CIBLE] if c in cols.columns]).copy()
    cat = [c for c in X.columns if not pd.api.types.is_numeric_dtype(X[c])]
    trop_fin = [c for c in cat if X[c].nunique() > 255]
    X = X.drop(columns=trop_fin)
    cat = [c for c in cat if c not in trop_fin]
    for c in cat:
        X[c] = X[c].fillna("").astype("category")
    return X, cat, trop_fin

X, CAT, TROP_FIN = matrice(train)
Xt, _, _ = matrice(test)

for c in CAT:
    Xt[c] = Xt[c].cat.set_categories(X[c].cat.categories)

print("la table de travail \n")
print("colonnes conservées :", X.shape[1])
print("colonnes écartées (cardinalité trop forte) :", TROP_FIN)
print("colonnes catégorielles :", CAT)


la table de travail 

colonnes conservées : 26
colonnes écartées (cardinalité trop forte) : ['tournee_jour']
colonnes catégorielles : ['clinique', 'district', 'sexe', 'quartier', 'assurance', 'rappel_envoye', 'zone_clinique']


### La cible, et la référence


In [6]:
print("la distribution de la cible \n")
repartition = pd.DataFrame({"lignes": [len(train), int(train[CIBLE].sum())],
                            "part (%)": [100.0, round(100 * train[CIBLE].mean(), 2)]},
                           index=["toutes", "absents"])
display(repartition)

reference = 100 * max(train[CIBLE].mean(), 1 - train[CIBLE].mean())
print(f"la référence (répondre toujours la classe majoritaire) : {reference:.2f} %")


la distribution de la cible 



,lignes,part (%)
toutes,149894,100.00
absents,27357,18.25


la référence (répondre toujours la classe majoritaire) : 81.75 %


In [7]:
# `y` entre ici ; la référence a déjà été calculée avec la distribution ci-dessus —
y = train[CIBLE]


### Vos features, et la chaîne


In [8]:
# votre feature : elle reçoit la TABLE DE TRAVAIL — identifiants, dates et colonnes post-issue
# déjà retirés — et la rend enrichie. `predict` l'appelle sur la table BRUTE, après avoir retiré
# les mêmes colonnes : le crochet voit donc la même forme des deux côtés, ce qui est la règle.
def ajouter_features(df):
    """Vos features maison. Appliquée identiquement à l'entraînement et au test."""
    df = df.copy()
    # Exemple — remplacez-le par la vôtre :
    # df["plus_de_60_ans"] = (df["age"] > 60).astype(int)
    return df

# Votre feature entre par `preparer()`, la chaîne unique : on n'applique RIEN à la main.
# Après l'avoir écrite, appelez `remesurer()` (au devoir, dans la cellule prévue) : `X` la
# reçoit une seule fois, et `predict` l'applique aussi, de son côté, sur la table brute.


In [9]:
# ── LA CHAÎNE : une seule fonction, celle que `predict` ET l'entraînement utilisent.
def preparer(brut):
    """Écarte, type, applique vos features : LA chaîne, des deux côtés.
       `predict` l'appelle ; l'entraînement l'appelle aussi, par `matrice()` sur la table de
       travail. Contourner cette fonction, c'est apprendre sur une table et prédire sur une
       autre — exactement le défaut que la séance 2 traque."""
    d = brut.drop(columns=[c for c in A_ECARTER + TROP_FIN + [CIBLE] if c in brut.columns],
                  errors="ignore").copy()
    for c in CAT:
        if c in d.columns:
            d[c] = d[c].fillna("").astype("category").cat.set_categories(X[c].cat.categories)
    return ajouter_features(d)

def remesurer():
    """Réentraîne sur la table de travail COURANTE et met `modele` à jour.
       `ajouter_features` n'est appliquée qu'UNE fois, depuis X : ré-exécutable sans que la
       feature se superpose à elle-même."""
    global modele
    Xc = ajouter_features(X)
    a, b, c, d = train_test_split(Xc, y, test_size=0.2, stratify=y, random_state=0)
    modele = HistGradientBoostingClassifier(random_state=0).fit(a, c)
    return 100 * modele.score(b, d)


### Le rituel : annoncer avant de mesurer


In [10]:
# l'horloge entre ici : c'est elle qui rend une prédiction vérifiable
from datetime import datetime

ESSAIS = []
PREDICTIONS = []

def prevoir(essai, attendu, raison):
    """Écrit ce que vous attendez AVANT de mesurer — horodaté.
       `noter(...)` refuse ensuite tout essai sans prédiction : l'ordre n'est pas demandé, il est
       appliqué. Ce que cette fonction garantit, exactement : une prédiction EXISTAIT avant que la
       ligne du tableau soit écrite. Elle ne date pas la mesure — le score lui arrive déjà calculé.
       Pour la garantie complète, mesurez dans `essai(...)`, qui date la mesure lui-même."""
    PREDICTIONS[:] = [p for p in PREDICTIONS if p["essai"] != essai]
    PREDICTIONS.append({"essai": essai, "attendu": attendu, "raison": raison,
                        "écrit à": datetime.now().strftime("%H:%M:%S")})
    display(pd.DataFrame(PREDICTIONS))

def essai(nom, mesure, changement="", attendu=""):
    """Le chemin STRICT : la mesure est prise ICI, donc l'ordre (prédiction puis mesure) est prouvé.
       `mesure` est une fonction sans argument qui rend le score, par exemple
       `lambda: modele.score(Xval, yval)`. Refuse de mesurer sans prédiction."""
    prevu = [p for p in PREDICTIONS if p["essai"] == nom]
    if not prevu:
        print(f"refusé : aucune prédiction pour « {nom} ». Écrivez prevoir(...) d'abord.")
        return None
    score = 100 * mesure()
    noter(nom, changement or "voir la question", attendu or prevu[0]["attendu"], score)
    return score

def noter(hypothese, changement, attendu, score):
    """Ajoute une ligne au tableau. `attendu` : un signe et une raison, jamais une valeur.
       Ré-exécuter une cellule REMPLACE la ligne de même `hypothèse` : elle ne la double pas.

       REFUSE d'enregistrer un essai non annoncé — c'est la règle du cours, appliquée.
       Ce qu'elle garantit, et rien de plus : une prédiction existait AVANT que cette ligne soit
       écrite. Elle ne date pas la mesure — d'où `essai(...)` pour la garantie complète."""
    prevu = [p for p in PREDICTIONS if p["essai"] == hypothese]
    if not prevu:
        print(f"  refusé : aucune prédiction pour « {hypothese} ».")
        print("  Écrivez prevoir(...) d'abord — l'ordre est vérifié, pas seulement demandé.")
        return
    ligne = {"hypothèse": hypothese, "changement": changement,
             "attendu": attendu, "mesuré": round(score, 2),
             "prédiction écrite avant ?": "oui, à " + prevu[0]["écrit à"]}
    ESSAIS[:] = [e for e in ESSAIS if e["hypothèse"] != hypothese] + [ligne]
    display(pd.DataFrame(ESSAIS))


### Le découpage, le modèle, les baselines


In [11]:
# les baselines, sur le même jeu de validation — et la barre qui décide
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

Xtr, Xval, ytr, yval = train_test_split(X, y, test_size=0.2, stratify=y, random_state=0)

# 1) la réponse constante : le concurrent le plus bête, et le plus dur à battre
constante = 100 * max(yval.mean(), 1 - yval.mean())

# 2) la règle du délai : « j'annonce une absence si le délai dépasse N jours ».
#    N se choisit sur les lignes d'apprentissage, pas sur celles de contrôle.
delai_max = int(Xtr['delai_jours'].max())
taux_par_seuil = {s: float(((Xtr['delai_jours'] > s).astype(int) == ytr).mean())
                  for s in (0, 1, 3, 7, 14, 30) if s < delai_max}
seuil_appris = max(taux_par_seuil, key=taux_par_seuil.get)
regle = (Xval['delai_jours'] > seuil_appris).astype(int)

# et la règle qui ne dirait jamais rien ? c'est la réponse constante, au point près
silence = (Xval['delai_jours'] > delai_max).astype(int)

# 3) la famille linéaire : elle a besoin de nombres, on lui en prépare — la recette des
#    séances précédentes (newton-cholesky : le solveur qui converge sur cette table)
X_n = pd.get_dummies(X, columns=CAT).fillna(0)   # 0 = trou comblé ; le linéaire ne lit pas les NaN
Xtr_n, Xval_n = X_n.loc[Xtr.index], X_n.loc[Xval.index]
lineaire = LogisticRegression(solver='newton-cholesky').fit(Xtr_n, ytr)

# 4) le modèle du cours, celui des séances précédentes
modele = HistGradientBoostingClassifier(random_state=0).fit(Xtr, ytr)

# la barre : l'erreur-type de l'exactitude sur CE jeu de validation
barre = 100 * ((constante / 100) * (1 - constante / 100) / len(yval)) ** 0.5

baselines = pd.DataFrame([
    {'baseline': 'toujours présent',
     'exactitude (%)': round(constante, 2)},
    {'baseline': f'règle du délai — absent si le délai dépasse {seuil_appris} j',
     'exactitude (%)': round(100 * (regle == yval).mean(), 2)},
    {'baseline': 'absent si ce patient a déjà manqué un rendez-vous',
     'exactitude (%)': round(100 * ((Xval['taux_absence_passe'] > 0).astype(int) == yval).mean(), 2)},
    {'baseline': 'la famille linéaire',
     'exactitude (%)': round(100 * lineaire.score(Xval_n, yval), 2)},
    {'baseline': 'le modèle du cours',
     'exactitude (%)': round(100 * modele.score(Xval, yval), 2)},
])
display(baselines)

ecart = 100 * modele.score(Xval, yval) - constante
print(f'→ le modèle dépasse la réponse constante de {ecart:+.2f} point,')
print(f'  soit {ecart / barre:.1f} erreurs-types de ce jeu de validation ({barre:.2f} point).')
print(f'  la règle du délai, elle, fait {100 * (regle == yval).mean():.2f} % : elle alerte sur')
print(f'  {int(regle.sum())} lignes et perd contre la réponse constante.')
print(f'  et la règle qui ne dirait jamais rien ferait {100 * (silence == yval).mean():.2f} % —')
print('  exactement la réponse constante. Une règle en place se mesure avant d’être opposée')
print('  à un modèle ; ce que cela dit d’une exactitude reste ouvert.')


,baseline,exactitude (%)
0,toujours présent,81.75
1,règle du délai — absent si le délai dépasse 30 j,79.92
2,absent si ce patient a déjà manqué un rendez-vous,78.65
3,la famille linéaire,82.21
4,le modèle du cours,82.51


→ le modèle dépasse la réponse constante de +0.76 point,
  soit 3.4 erreurs-types de ce jeu de validation (0.22 point).
  la règle du délai, elle, fait 79.92 % : elle alerte sur
  1572 lignes et perd contre la réponse constante.
  et la règle qui ne dirait jamais rien ferait 81.75 % —
  exactement la réponse constante. Une règle en place se mesure avant d’être opposée
  à un modèle ; ce que cela dit d’une exactitude reste ouvert.


### La matrice, et les probabilités


In [12]:
# les quatre cases, comptées à la main sur le jeu de contrôle
predit = modele.predict(Xval)

VP = int(((predit == 1) & (yval == 1)).sum())   # annoncé absent, et absent
FP = int(((predit == 1) & (yval == 0)).sum())   # annoncé absent, et venu : la fausse alerte
FN = int(((predit == 0) & (yval == 1)).sum())   # annoncé présent, et absent : l’absence manquée
VN = int(((predit == 0) & (yval == 0)).sum())   # annoncé présent, et venu

assert VP + FP + FN + VN == len(yval)

# et la probabilité, qui servira à déplacer le seuil : c’est elle qui décide, pas le 0/1
probabilites = modele.predict_proba(Xval)[:, 1]

# les quatre mêmes cases, retrouvées par la bibliothèque — le contrôle, pas la confiance
from sklearn.metrics import confusion_matrix
confirme = confusion_matrix(yval, predit).ravel()          # ordre : VN, FP, FN, VP
assert (VN, FP, FN, VP) == tuple(int(v) for v in confirme), 'les quatre cases ne concordent pas'
print(f'les quatre cases, vérifiées : VP {VP} · FP {FP} · FN {FN} · VN {VN}')
print(f'→ {FP + FN} erreurs en tout : {FP} fausses alertes, {FN} absences manquées.')


les quatre cases, vérifiées : VP 580 · FP 353 · FN 4891 · VN 24155
→ 5244 erreurs en tout : 353 fausses alertes, 4891 absences manquées.


### Le coût, et sa formule


In [13]:
# le coût d’une décision, et la dérivation du seuil
COUT_FP, COUT_FN = 25, 100   # les deux montants de l’énoncé, en unités de coût

def cout(decision, verite=yval):
    faux_positifs = int(((decision == 1) & (verite == 0)).sum())
    faux_negatifs = int(((decision == 0) & (verite == 1)).sum())
    return COUT_FP * faux_positifs + COUT_FN * faux_negatifs

seuil_theorique = COUT_FP / (COUT_FP + COUT_FN)
print('la dérivation du seuil \n')
print(f'  pour un patient de probabilité p, alerter coûte 25 × (1 − p) ; ne pas alerter, 100 × p')
print(f'  on alerte tant que 25 × (1 − p) < 100 × p, soit p > 25 / (25 + 100) = {seuil_theorique:.2f}')
print('  → ce chiffre suppose que les probabilités annoncées veulent dire quelque chose.')


la dérivation du seuil 

  pour un patient de probabilité p, alerter coûte 25 × (1 − p) ; ne pas alerter, 100 × p
  on alerte tant que 25 × (1 − p) < 100 × p, soit p > 25 / (25 + 100) = 0.20
  → ce chiffre suppose que les probabilités annoncées veulent dire quelque chose.


### Les quatre objets que la séance 4 avait mesurés

La séance 4 a déjà construit la courbe du coût, la formule du seuil, le seuil appris et le
tableau des stratégies. Les recalculer ne vous apprendrait rien de plus : ils sont donnés.

**`courbe`** — une ligne par seuil, au pas de 0,02, sur `[0,04 ; 0,96]` : les alertes, l'exactitude et le coût.
**`seuil_theorique`** — `COUT_FP / (COUT_FP + COUT_FN)`.
**`seuil_appris_cout`** — le seuil choisi par validation croisée sur l'apprentissage, donc hors de la mesure de contrôle.
**`lignes`** — neuf stratégies avec leur exactitude et leur coût, **non triées**.

La cellule suivante les construit une seule fois. Elle entraîne un modèle en validation
croisée : comptez une à trois minutes, et ne la relancez pas sans raison.


In [14]:
# les quatre objets donnés — construits une seule fois, à partir de la chaîne du cours
import numpy as np
import pandas as pd

SEUILS = [0.10, 0.20, 0.30, 0.40, 0.50, 0.60]
PAS = np.round(np.arange(0.04, 0.97, 0.02), 2)


def decision_au(seuil):
    """La décision à un seuil : 1 = on annonce une absence."""
    return (probabilites > seuil).astype(int)


courbe = pd.DataFrame([
    {'seuil': float(s),
     'alertes': int(decision_au(s).sum()),
     'exactitude (%)': round(100 * (decision_au(s) == yval).mean(), 2),
     'coût': cout(decision_au(s))}
    for s in PAS])

# le seuil appris choisit sur l'apprentissage, jamais sur le contrôle
from sklearn.model_selection import cross_val_predict
proba_apprises = cross_val_predict(
    HistGradientBoostingClassifier(random_state=0), Xtr, ytr, cv=5,
    method='predict_proba')[:, 1]
seuil_appris_cout = float(min(PAS, key=lambda s: cout((proba_apprises > s).astype(int),
                                                      verite=ytr)))

strategies_donnees = {'n’alerte jamais': np.zeros(len(yval), dtype=int),
                      'alerte toujours': np.ones(len(yval), dtype=int)}
for s in SEUILS:
    strategies_donnees[f'le modèle, seuil {s:.2f}'] = decision_au(s)
strategies_donnees[f'seuil appris ({seuil_appris_cout:.2f})'] = decision_au(seuil_appris_cout)

lignes = pd.DataFrame([
    {'stratégie': nom,
     'exactitude (%)': round(100 * (d == yval).mean(), 2),
     'coût': cout(d)}
    for nom, d in strategies_donnees.items()])

# la valeur que le prédicteur constant obtient sur CHAQUE métrique — pas une seule pour toutes
# il n'alerte jamais : rappel nul, spécificité pleine, précision indéfinie (0/0, comptée 0)
reference_par_metrique = {'exactitude': float(constante),
                          'précision': 0.0,
                          'rappel': 0.0,
                          'spécificité': 100.0,
                          'F1': 0.0,
                          'exactitude équilibrée': 50.0}

display(lignes)
print('→ neuf stratégies, deux mesures, aucune triée : le tri est votre travail.')


,stratégie,exactitude (%),coût
0,n’alerte jamais,81.75,547100
1,alerte toujours,18.25,612700
2,"le modèle, seuil 0.10",50.92,408900
3,"le modèle, seuil 0.20",67.02,372850
4,"le modèle, seuil 0.30",78.72,421800
5,"le modèle, seuil 0.40",82.11,469325
6,"le modèle, seuil 0.50",82.51,497925
7,"le modèle, seuil 0.60",82.45,514975
8,seuil appris (0.18),63.72,372700


→ neuf stratégies, deux mesures, aucune triée : le tri est votre travail.


# Volet A — la trajectoire des six métriques

**Contexte.** Une métrique mesurée à un seul seuil ne dit pas ce qu'elle récompense : elle dit
ce qu'elle valait ce jour-là. Ici, on regarde ce que chaque métrique fait **quand le seuil
bouge**.


## A1 — les six métriques, à six seuils

**Consigne.** Construisez `mon_tableau_seuils` : exactitude, précision, rappel, spécificité, F1
et exactitude équilibrée, chacune calculée aux six seuils de `SEUILS`.

Une ligne par métrique, une colonne par seuil, en pourcentage. Le seuil 0,50 est celui que le
modèle applique par défaut : il sert de point de repère, pas de réponse.

**Attention à deux cas.** Une précision s'écrit `0/0` quand le modèle n'alerte jamais : ce
n'est pas zéro. Et F1 divisé par une somme nulle n'existe pas.

**Livrable.** `mon_tableau_seuils` — un tableau 6 × 6.
**Critère.** Deux métriques au même seuil ne mesurent pas la même chose, et l'une des six
peut être flatteuse sur une décision inutile.


In [34]:
from sklearn.metrics import confusion_matrix

resultats = {}

for seuil in SEUILS:
    prediction = (probabilites > seuil).astype(int)

    VN, FP, FN, VP = confusion_matrix(
        yval, prediction, labels=[0, 1]
    ).ravel()

    exactitude = (VP + VN) / (VP + VN + FP + FN) * 100

    precision = VP / (VP + FP) * 100 if (VP + FP) > 0 else 0

    rappel = VP / (VP + FN) * 100 if (VP + FN) > 0 else 0

    specificite = VN / (VN + FP) * 100 if (VN + FP) > 0 else 0

    f1 = (2 * VP / (2 * VP + FP + FN)) * 100 if (2 * VP + FP + FN) > 0 else 0

    equilibree = (rappel + specificite) / 2

    resultats[seuil] = [
        exactitude,
        precision,
        rappel,
        specificite,
        f1,
        equilibree
    ]

mon_tableau_seuils = pd.DataFrame(
    resultats,
    index=[
        'exactitude',
        'précision',
        'rappel',
        'spécificité',
        'F1',
        'exactitude équilibrée'
    ]
)

mon_tableau_seuils = mon_tableau_seuils.round(2)

display(mon_tableau_seuils)

,0.1,0.2,0.3,0.4,0.5,0.6
exactitude,50.92,67.02,78.72,82.11,82.51,82.45
précision,25.79,31.61,40.63,52.85,62.17,70.89
rappel,90.00,69.37,36.08,18.30,10.60,6.54
spécificité,42.19,66.50,88.23,96.36,98.56,99.40
F1,40.09,43.43,38.22,27.18,18.11,11.98
exactitude équilibrée,66.10,67.93,62.16,57.33,54.58,52.97


In [15]:
controle('`mon_tableau_seuils` a six lignes et six colonnes',
         lambda: mon_tableau_seuils.shape == (6, 6),
         'six métriques aux six seuils de SEUILS')
controle('les six noms de métriques sont ceux de la séance',
         lambda: set(map(str, mon_tableau_seuils.index)) ==
                 {'exactitude', 'précision', 'rappel', 'spécificité', 'F1',
                  'exactitude équilibrée'},
         'les six métriques nommées en toutes lettres')
controle('chaque case est un pourcentage, sans trou',
         lambda: mon_tableau_seuils.notna().all().all()
                 and float(mon_tableau_seuils.min().min()) >= 0
                 and float(mon_tableau_seuils.max().max()) <= 100,
         'entre 0 et 100, et aucune case vide')
controle('les colonnes sont les six seuils, dans l ordre',
         lambda: [float(c) for c in mon_tableau_seuils.columns] == SEUILS,
         'les colonnes suivent SEUILS, de 0,10 à 0,60')


  (pas encore écrit)  `mon_tableau_seuils` a six lignes et six colonnes
  (pas encore écrit)  les six noms de métriques sont ceux de la séance
  (pas encore écrit)  chaque case est un pourcentage, sans trou
  (pas encore écrit)  les colonnes sont les six seuils, dans l ordre


## A2 — le seuil qui maximise chaque métrique

**Consigne.** Pour chacune des six métriques, donnez le seuil de `SEUILS` qui la maximise.

**Livrable.** `seuil_gagnant` — un dictionnaire, nom de métrique vers seuil.
**Critère.** Deux métriques peuvent désigner deux seuils différents. C'est le résultat, pas
une erreur.


In [35]:
seuil_gagnant = {}

for metrique in mon_tableau_seuils.index:
    seuil_gagnant[metrique] = mon_tableau_seuils.loc[metrique].idxmax()

print(seuil_gagnant)

{'exactitude': np.float64(0.5), 'précision': np.float64(0.6), 'rappel': np.float64(0.1), 'spécificité': np.float64(0.6), 'F1': np.float64(0.2), 'exactitude équilibrée': np.float64(0.2)}


## A3 — combien de maxima tombent au seuil par défaut

**Consigne.** Comptez combien des six maxima tombent à `0,50` (`combien_a_050`), et écrivez en
une phrase ce que ce chiffre dit du seuil par défaut (`lecture_du_defaut`).

**Livrable.** `combien_a_050` (un entier de 0 à 6) et `lecture_du_defaut` (une phrase).
**Critère.** Un seuil par défaut n'est pas un seuil choisi : il est le point où l'on n'a rien
décidé.


In [36]:
combien_a_050 = sum(
    seuil == 0.50 for seuil in seuil_gagnant.values()
)

lecture_du_defaut = (
    "Le seuil de 0,50 maximise uniquement l'exactitude ; "
    "il ne constitue donc pas le meilleur choix pour toutes les métriques."
)

print(combien_a_050)
print(lecture_du_defaut)

1
Le seuil de 0,50 maximise uniquement l'exactitude ; il ne constitue donc pas le meilleur choix pour toutes les métriques.


In [16]:
controle('`seuil_gagnant` couvre les six métriques',
         lambda: set(seuil_gagnant) == set(map(str, mon_tableau_seuils.index)),
         'les six noms, exactement')
controle('chaque seuil gagnant est l un des six seuils',
         lambda: all(float(v) in SEUILS for v in seuil_gagnant.values()),
         'un seuil de SEUILS, pas une valeur inventée')
controle('chaque seuil maximise bien sa métrique dans votre tableau',
         lambda: all(float(seuil_gagnant[m]) ==
                     float(mon_tableau_seuils.columns[
                         list(mon_tableau_seuils.loc[m]).index(
                             max(mon_tableau_seuils.loc[m]))])
                     for m in mon_tableau_seuils.index),
         'le maximum de la ligne, pas un autre seuil')


  (pas encore écrit)  `seuil_gagnant` couvre les six métriques
  (pas encore écrit)  chaque seuil gagnant est l un des six seuils
  (pas encore écrit)  chaque seuil maximise bien sa métrique dans votre tableau


## A4 — l'amplitude : ce que chaque métrique gagne à bouger

**Consigne.** Pour chaque métrique, calculez l'écart entre son maximum et son minimum sur les
six seuils. Nommez ensuite celle des six qui **bouge le moins**.

Une métrique qui bouge peu ne récompense pas un choix de seuil : elle donne le même chiffre
quelle que soit la décision, et c'est un défaut pour arbitrer.

**Livrable.** `amplitude` — un dictionnaire, nom de métrique vers écart en points — et
`moins_sensible` — le nom de la métrique d'amplitude minimale.
**Critère.** Laquelle des six est la plus indifférente au seuil, et ce que cela implique pour
qui veut choisir un seuil.


In [37]:
amplitude = {}

for metrique in mon_tableau_seuils.index:
    amplitude[metrique] = (
        mon_tableau_seuils.loc[metrique].max()
        - mon_tableau_seuils.loc[metrique].min()
    )

moins_sensible = min(amplitude, key=amplitude.get)

print("Amplitudes :", amplitude)
print("Métrique la moins sensible :", moins_sensible)

Amplitudes : {'exactitude': 31.590000000000003, 'précision': 45.1, 'rappel': 83.46, 'spécificité': 57.21000000000001, 'F1': 31.45, 'exactitude équilibrée': 14.960000000000008}
Métrique la moins sensible : exactitude équilibrée


In [17]:
controle('`combien_a_050` est un compte, entre 0 et 6',
         lambda: isinstance(combien_a_050, (int, np.integer))
                 and 0 <= int(combien_a_050) <= 6,
         'un entier : un nombre de métriques, pas un pourcentage')
controle('`lecture_du_defaut` est une phrase, pas un chiffre',
         lambda: isinstance(lecture_du_defaut, str) and len(lecture_du_defaut) > 40,
         'ce que le chiffre dit du seuil par défaut')
controle('les deux réponses se répondent',
         lambda: str(int(combien_a_050)) in lecture_du_defaut
                 or (int(combien_a_050) == 0 and 'aucun' in lecture_du_defaut.lower()),
         'la phrase cite le compte qu elle commente')


  (pas encore écrit)  `combien_a_050` est un compte, entre 0 et 6
  (pas encore écrit)  `lecture_du_defaut` est une phrase, pas un chiffre
  (pas encore écrit)  les deux réponses se répondent


In [18]:
controle('`amplitude` couvre les six métriques',
         lambda: set(amplitude) == set(map(str, mon_tableau_seuils.index)),
         'les six noms, exactement')
controle('chaque amplitude est un écart, positif ou nul',
         lambda: all(float(v) >= 0 for v in amplitude.values()),
         'un maximum moins un minimum : jamais négatif')
controle('l amplitude est cohérente avec votre tableau',
         lambda: all(abs(float(amplitude[m]) -
                         (float(max(mon_tableau_seuils.loc[m])) -
                          float(min(mon_tableau_seuils.loc[m])))) < 0.01
                     for m in mon_tableau_seuils.index),
         'max moins min, sur votre propre tableau')
controle('`moins_sensible` est bien la plus petite amplitude',
         lambda: moins_sensible in amplitude
                 and float(amplitude[moins_sensible]) ==
                     min(float(v) for v in amplitude.values()),
         'le nom de la métrique dont l amplitude est minimale')


  (pas encore écrit)  `amplitude` couvre les six métriques
  (pas encore écrit)  chaque amplitude est un écart, positif ou nul
  (pas encore écrit)  l amplitude est cohérente avec votre tableau
  (pas encore écrit)  `moins_sensible` est bien la plus petite amplitude


## A5 — la rupture : quand une métrique cesse de battre la référence

**Contexte.** La référence de la séance 3 est le prédicteur constant : celui qui répond
toujours la classe majoritaire. Mais ce prédicteur n'obtient pas le même score sur chaque
métrique — il n'alerte jamais, donc son rappel est nul et sa spécificité pleine. Comparer un
rappel à une exactitude serait comparer deux unités différentes.

`reference_par_metrique` porte donc, pour chaque métrique, ce que le prédicteur constant y
obtient.

**Consigne.** Pour chaque métrique, donnez le premier seuil de `SEUILS` où elle **dépasse sa
propre référence**. Si elle ne la dépasse à aucun des six, écrivez `aucun`.

**Livrable.** `rupture` — un dictionnaire, nom de métrique vers seuil ou `aucun`.
**Critère.** Une métrique que le prédicteur constant maximise déjà ne peut rien vous apprendre
sur le modèle — et le tableau le dit, chiffre en main.


In [38]:
rupture = {}

for metrique in mon_tableau_seuils.index:
    rupture[metrique] = "aucun"

    for seuil in SEUILS:
        score = mon_tableau_seuils.loc[metrique, seuil]
        reference = reference_par_metrique[metrique]

        if score > reference:
            rupture[metrique] = seuil
            break

print(rupture)

{'exactitude': 0.4, 'précision': 0.1, 'rappel': 0.1, 'spécificité': 'aucun', 'F1': 0.1, 'exactitude équilibrée': 0.1}


In [19]:

controle('chaque entrée est un seuil de SEUILS, ou le mot aucun',
         lambda: all((v == 'aucun') or (float(v) in SEUILS) for v in rupture.values()),
         'un des six seuils, ou aucun')
controle('la rupture est cohérente avec votre tableau et SA référence',
         lambda: all(
             (rupture[m] == 'aucun')
             or float(mon_tableau_seuils.loc[m, rupture[m]]) > float(reference_par_metrique[m])
             for m in mon_tableau_seuils.index),
         'au seuil annoncé, la métrique dépasse sa propre référence')
controle('la première rupture est bien la première',
         lambda: all(
             (rupture[m] == 'aucun')
             or all(float(mon_tableau_seuils.loc[m, s]) <= float(reference_par_metrique[m])
                    for s in SEUILS[:SEUILS.index(float(rupture[m]))])
             for m in mon_tableau_seuils.index),
         'aucun seuil antérieur ne dépasse déjà la référence')


  (pas encore écrit)  `rupture` couvre les six métriques
  (pas encore écrit)  chaque entrée est un seuil de SEUILS, ou le mot aucun
  (pas encore écrit)  la rupture est cohérente avec votre tableau et SA référence
  (pas encore écrit)  la première rupture est bien la première


## A6 — votre échantillon contre le jeu complet

**Consigne.** Les six métriques ont été mesurées sur tout `yval`. Refaites-les au seuil `0,50`
sur un sous-échantillon tiré avec `np.random.RandomState(0)` de **5 000** lignes prises dans
`yval`, puis divisez le plus grand écart observé par `barre`.

Répondez alors : cet écart dépasse-t-il **une** barre de bruit ?

**Livrable.** `ecart_perso_complet` — l'écart maximal, en barres — et `ecart_depasse` — un
booléen.
**Critère.** Un échantillon qui s'écarte d'une barre n'est pas faux : il est petit. Le dire
fait partie de la réponse.


In [39]:

tirage = np.random.RandomState(0).choice(
    len(yval), size=5000, replace=False
)

yval_5000 = yval.iloc[tirage]
proba_5000 = np.asarray(probabilites)[tirage]


prediction_5000 = (proba_5000 > 0.50).astype(int)

VN, FP, FN, VP = confusion_matrix(
    yval_5000, prediction_5000, labels=[0, 1]
).ravel()

exactitude = (VP + VN) / 5000 * 100
precision = VP / (VP + FP) * 100 if VP + FP > 0 else 0
rappel = VP / (VP + FN) * 100 if VP + FN > 0 else 0
specificite = VN / (VN + FP) * 100 if VN + FP > 0 else 0
f1 = 2 * VP / (2 * VP + FP + FN) * 100 if 2 * VP + FP + FN > 0 else 0
equilibree = (rappel + specificite) / 2

metriques_5000 = {
    'exactitude': exactitude,
    'précision': precision,
    'rappel': rappel,
    'spécificité': specificite,
    'F1': f1,
    'exactitude équilibrée': equilibree
}


metriques_completes = dict(zip(
    mon_tableau_seuils.index,
    resultats[0.5]
))

ecarts = {
    nom: abs(metriques_5000[nom] - metriques_completes[nom])
    for nom in metriques_5000
}

plus_grand_ecart = max(ecarts.values())

ecart_perso_complet = plus_grand_ecart / barre
ecart_depasse = ecart_perso_complet > 1

print("Métriques sur 5 000 lignes :", metriques_5000)
print("Écarts :", ecarts)
print("Écart en barres :", ecart_perso_complet)
print("Dépasse une barre de bruit :", ecart_depasse)

Métriques sur 5 000 lignes : {'exactitude': np.float64(82.24000000000001), 'précision': np.float64(63.94557823129252), 'rappel': np.float64(10.118406889128096), 'spécificité': np.float64(98.69810857283224), 'F1': np.float64(17.472118959107807), 'exactitude équilibrée': np.float64(54.40825773098017)}
Écarts : {'exactitude': np.float64(0.26775542880014314), 'précision': np.float64(1.780519281667658), 'rappel': np.float64(0.4829456972363708), 'spécificité': np.float64(0.1384545822985359), 'F1': np.float64(0.6415599915480321), 'exactitude équilibrée': np.float64(0.17224555746891923)}
Écart en barres : 7.981510716466662
Dépasse une barre de bruit : True


In [20]:
controle('`ecart_perso_complet` est un nombre de barres',
         lambda: isinstance(ecart_perso_complet, (int, float, np.floating))
                 and float(ecart_perso_complet) >= 0,
         'un écart divisé par la barre de bruit')
controle('`ecart_depasse` est un booléen',
         lambda: isinstance(ecart_depasse, (bool, np.bool_)),
         'vrai ou faux, jamais un nombre')
controle('les deux réponses concordent avec la barre',
         lambda: bool(ecart_depasse) == (float(ecart_perso_complet) > 1.0),
         'dépasser veut dire strictement plus d une barre')


  (pas encore écrit)  `ecart_perso_complet` est un nombre de barres
  (pas encore écrit)  `ecart_depasse` est un booléen
  (pas encore écrit)  les deux réponses concordent avec la barre


# Volet B — le coût, le seuil, et la facture

**Contexte.** La mise en place fixe `COUT_FP = 25` et `COUT_FN = 100`. La compétition, elle,
classe sur l'**exactitude**. La métrique affichée n'est pas la facture.


## B1 — le seuil que la courbe désigne

**Consigne.** Dans `courbe`, trouvez le seuil qui minimise le coût mesuré (`seuil_mesure`).
Comparez-le à `seuil_theorique`, et dites de combien ils diffèrent (`ecart_des_seuils`).

Puis écrivez, en une phrase, ce que la formule suppose pour être juste
(`hypothese_de_la_formule`).

**Livrable.** `seuil_mesure`, `ecart_des_seuils`, `hypothese_de_la_formule`.
**Critère.** La formule suppose que les probabilités annoncées veulent dire quelque chose. Si
elles sont mal calibrées, le seuil qu'elle donne est faux, et la courbe le montre.


In [40]:
display(courbe.head())
print("Colonnes :", courbe.columns)
print("Seuil théorique :", seuil_theorique)

,seuil,alertes,exactitude (%),coût
0,0.04,26753,28.27,545925
1,0.06,22322,41.72,460025
2,0.08,20184,47.95,423575
3,0.10,19092,50.92,408900
4,0.12,18048,53.71,395675


Colonnes : Index(['seuil', 'alertes', 'exactitude (%)', 'coût'], dtype='object')
Seuil théorique : 0.2


In [41]:

ligne_minimale = courbe.loc[courbe['coût'].idxmin()]

seuil_mesure = ligne_minimale['seuil']

print("Seuil qui minimise le coût :", seuil_mesure)
print("Coût minimal :", ligne_minimale['coût'])

Seuil qui minimise le coût : 0.18
Coût minimal : 372700.0


In [42]:
ecart_des_seuils = abs(seuil_mesure - seuil_theorique)

print("Écart entre les deux seuils :", ecart_des_seuils)

Écart entre les deux seuils : 0.020000000000000018


In [43]:
hypothese_de_la_formule = (
    "La formule suppose que les probabilités prédites par le modèle "
    "sont bien calibrées, c'est-à-dire qu'elles correspondent aux "
    "fréquences réelles des événements."
)

print(hypothese_de_la_formule)

La formule suppose que les probabilités prédites par le modèle sont bien calibrées, c'est-à-dire qu'elles correspondent aux fréquences réelles des événements.


In [44]:
print(seuil_mesure)
print(ecart_des_seuils)


0.18
0.020000000000000018


In [21]:
controle('`seuil_mesure` est bien le minimum de la courbe',
         lambda: float(seuil_mesure) ==
                     float(courbe.loc[courbe['coût'].idxmin(), 'seuil']),
         'le seuil de la ligne de coût minimal')
controle('`ecart_des_seuils` est un écart, jamais négatif',
         lambda: abs(float(ecart_des_seuils)) ==
                     abs(float(seuil_mesure) - float(seuil_theorique)),
         'seuil mesuré moins seuil de la formule')
controle('`hypothese_de_la_formule` nomme la calibration ou les probabilités',
         lambda: isinstance(hypothese_de_la_formule, str)
                 and len(hypothese_de_la_formule) > 40,
         'ce que la formule suppose pour être juste')


  (pas encore écrit)  `seuil_mesure` est bien le minimum de la courbe
  (pas encore écrit)  `ecart_des_seuils` est un écart, jamais négatif
  (pas encore écrit)  `hypothese_de_la_formule` nomme la calibration ou les probabilités


## B2 — plateau ou arête

**Consigne.** Calculez le coût à `seuil_mesure ± 0,05` et à `seuil_mesure ± 0,10`.
Rangez les quatre chiffres dans `sensibilite`.

Puis tranchez : votre seuil est-il un **plateau** (l'optimum est large, le choix est robuste)
ou une **arête** (on perd gros à cinq centièmes près) ? Écrivez le mot dans
`plateau_ou_arete`.

**Livrable.** `sensibilite` et `plateau_ou_arete`.
**Critère.** Un seuil optimal qui n'est pas un plateau n'est pas une décision, c'est une
coïncidence.


In [45]:

deplacements = [-0.10, -0.05, 0.05, 0.10]

sensibilite = {}

for d in deplacements:
    seuil_test = seuil_mesure + d

    ligne = courbe.loc[
        (courbe['seuil'] - seuil_test).abs().idxmin()
    ]

    sensibilite[round(d, 2)] = ligne['coût']

print("Sensibilité :", sensibilite)

Sensibilité : {-0.1: np.float64(423575.0), -0.05: np.float64(395675.0), 0.05: np.float64(377125.0), 0.1: np.float64(410200.0)}


In [46]:
if sensibilite[0.05] < courbe['coût'].min() * 1.05:
    plateau_ou_arete = "plateau"
else:
    plateau_ou_arete = "arête"

print("Verdict :", plateau_ou_arete)

Verdict : plateau


In [22]:
controle('`sensibilite` porte les quatre déplacements',
         lambda: set(sensibilite) ==
                 {-0.10, -0.05, 0.05, 0.10},
         'quatre clés : −0,10 · −0,05 · +0,05 · +0,10')
controle('chaque coût est un nombre positif',
         lambda: all(float(v) >= 0 for v in sensibilite.values()),
         'un coût se compte en unités, jamais en pourcentage')
controle('`plateau_ou_arete` est l un des deux mots',
         lambda: str(plateau_ou_arete).strip().lower() in
                 {'plateau', 'arête', 'arete'},
         'plateau ou arête : pas une phrase')
controle('le verdict suit la sensibilité mesurée',
         lambda: (str(plateau_ou_arete).strip().lower() != 'plateau')
                 or (float(sensibilite[0.05]) < float(courbe['coût'].min()) * 1.05),
         'un plateau tolère cinq centièmes ; une arête non')


  (pas encore écrit)  `sensibilite` porte les quatre déplacements
  (pas encore écrit)  chaque coût est un nombre positif
  (pas encore écrit)  `plateau_ou_arete` est l un des deux mots
  (pas encore écrit)  le verdict suit la sensibilité mesurée


## B3 — la facture, en trois unités

**Consigne.** `cout_a_050` est le coût au seuil par défaut, `cout_a_mon_seuil` le coût au seuil
que vous retenez. Leur différence est `facture_de_la_confiance`.

Rendez cette facture **trois fois** :
1. en unités de coût — `facture_unites` ;
2. en **créneaux équivalents**, en divisant par `COUT_FN` — `facture_creneaux` ;
3. en pourcentage du coût au seuil par défaut — `facture_pourcent`.

Un créneau équivalent est une consultation manquée : c'est l'unité que la direction de la
clinique comprend sans tableur.

**Livrable.** `cout_a_050`, `cout_a_mon_seuil`, `facture_de_la_confiance`, `facture_unites`,
`facture_creneaux`, `facture_pourcent`.
**Critère.** La même perte en trois langues. Une seule ne suffit pas : le coût parle au
gestionnaire, le créneau au soignant, le pourcentage au rapport.


In [47]:

cout_a_050 = courbe.loc[
    courbe['seuil'] == 0.50, 'coût'
].iloc[0]


cout_a_mon_seuil = cout(
    (probabilites > seuil_mesure).astype(int)
)


facture_de_la_confiance = cout_a_050 - cout_a_mon_seuil


facture_unites = facture_de_la_confiance
facture_creneaux = facture_de_la_confiance / COUT_FN
facture_pourcent = (
    100 * facture_de_la_confiance / cout_a_050
)

print("Coût à 0,50 :", cout_a_050)
print("Coût à 0,18 :", cout_a_mon_seuil)
print("Facture en unités :", facture_unites)
print("Facture en créneaux :", facture_creneaux)
print("Facture en pourcentage :", facture_pourcent, "%")

Coût à 0,50 : 497925
Coût à 0,18 : 372700
Facture en unités : 125225
Facture en créneaux : 1252.25
Facture en pourcentage : 25.149369885022846 %


In [23]:
controle('`cout_a_050` est le coût au seuil par défaut',
         lambda: abs(float(cout_a_050) -
                     float(courbe.loc[courbe['seuil'] == 0.50, 'coût'].iloc[0])) < 1,
         'la ligne du seuil 0,50 dans courbe')
controle('`cout_a_mon_seuil` est le coût au seuil que vous retenez',
         lambda: abs(float(cout_a_mon_seuil) -
                     float(cout((probabilites > seuil_mesure).astype(int)))) < 1,
         'le coût recalculé à votre seuil')
controle('la facture est la différence des deux coûts',
         lambda: abs(float(facture_de_la_confiance) -
                     (float(cout_a_050) - float(cout_a_mon_seuil))) < 1,
         'coût à 0,50 moins coût à votre seuil')
controle('la facture en unités vaut la facture',
         lambda: abs(float(facture_unites) - float(facture_de_la_confiance)) < 1,
         'la même grandeur, dite en unités de coût')
controle('les créneaux équivalents divisent par COUT_FN',
         lambda: abs(float(facture_creneaux) -
                     float(facture_de_la_confiance) / COUT_FN) < 0.01,
         'une absence manquée = COUT_FN unités')
controle('le pourcentage se rapporte au coût à 0,50',
         lambda: abs(float(facture_pourcent) -
                     100 * float(facture_de_la_confiance) / float(cout_a_050)) < 0.01,
         'part du coût par défaut que la facture représente')


  (pas encore écrit)  `cout_a_050` est le coût au seuil par défaut
  (pas encore écrit)  `cout_a_mon_seuil` est le coût au seuil que vous retenez
  (pas encore écrit)  la facture est la différence des deux coûts
  (pas encore écrit)  la facture en unités vaut la facture
  (pas encore écrit)  les créneaux équivalents divisent par COUT_FN
  (pas encore écrit)  le pourcentage se rapporte au coût à 0,50


## B4 — la sensibilité au rapport de coûts

**Contexte.** Les deux montants `25` et `100` sortent de l'énoncé, pas d'une mesure. Une
décision qui tient seulement à ce rapport tient à une hypothèse.

**Consigne.** Recalculez le seuil de coût minimal pour trois rapports : **1:1** (`100` et
`100`), **4:1** (le rapport du cours, `25` et `100`) et **10:1** (`10` et `100`). Rangez les
trois seuils dans `sensibilite_au_rapport`, et dites si le classement des trois seuils est
robuste (`rapport_robuste`).

**Livrable.** `sensibilite_au_rapport` (rapport vers seuil) et `rapport_robuste` (booléen).
**Critère.** Si le seuil se déplace de plus de dix centièmes entre 1:1 et 10:1, la décision
dépend du prix, et le prix n'a pas été mesuré.


In [48]:

rapports = {
    '1:1': (100, 100),
    '4:1': (25, 100),
    '10:1': (10, 100)
}

sensibilite_au_rapport = {}

for rapport, (COUT_FP, COUT_FN) in rapports.items():

    
    couts = []

    for seuil in courbe['seuil']:
        prediction = (probabilites > seuil).astype(int)

        VN, FP, FN, VP = confusion_matrix(
            yval, prediction, labels=[0, 1]
        ).ravel()

        cout_total = FP * COUT_FP + FN * COUT_FN
        couts.append(cout_total)

    
    indice_min = np.argmin(couts)
    seuil_optimal = courbe['seuil'].iloc[indice_min]

    sensibilite_au_rapport[rapport] = seuil_optimal

print("Seuils selon les rapports :", sensibilite_au_rapport)

Seuils selon les rapports : {'1:1': np.float64(0.52), '4:1': np.float64(0.18), '10:1': np.float64(0.08)}


In [49]:
seuils = list(sensibilite_au_rapport.values())

rapport_robuste = (max(seuils) - min(seuils)) <= 0.10

print("Décision robuste :", rapport_robuste)

Décision robuste : False


In [24]:
controle('`sensibilite_au_rapport` porte les trois rapports',
         lambda: set(map(str, sensibilite_au_rapport)) ==
                 {'1:1', '4:1', '10:1'},
         'trois clés : 1:1, 4:1, 10:1')
controle('chaque seuil est dans la plage de la courbe',
         lambda: all(0.0 <= float(v) <= 1.0 for v in sensibilite_au_rapport.values()),
         'un seuil de probabilité, entre 0 et 1')
controle('le rapport du cours redonne votre seuil mesuré',
         lambda: abs(float(sensibilite_au_rapport['4:1']) - float(seuil_mesure)) < 0.021,
         'à 25 contre 100, on retrouve le seuil de la courbe')
controle('`rapport_robuste` est un booléen',
         lambda: isinstance(rapport_robuste, (bool, np.bool_)),
         'vrai ou faux')
controle('la robustesse suit l écart mesuré',
         lambda: bool(rapport_robuste) ==
                 (max(float(v) for v in sensibilite_au_rapport.values())
                  - min(float(v) for v in sensibilite_au_rapport.values()) <= 0.10),
         'robuste veut dire dix centièmes d écart au plus')


  (pas encore écrit)  `sensibilite_au_rapport` porte les trois rapports
  (pas encore écrit)  chaque seuil est dans la plage de la courbe
  (pas encore écrit)  le rapport du cours redonne votre seuil mesuré
  (pas encore écrit)  `rapport_robuste` est un booléen
  (pas encore écrit)  la robustesse suit l écart mesuré


# Volet C — les deux classements, exhaustifs

**Contexte.** La compétition classe sur l'exactitude ; la décision se paie en coût. `lignes`
porte les neuf stratégies et leurs deux mesures, **non triées**.


## C1 — les deux classements

**Consigne.** Triez `lignes` deux fois : par exactitude décroissante, puis par coût croissant.
Rangez le résultat dans deux dictionnaires, `rang_exactitude` et `rang_cout`, où chaque
stratégie reçoit son rang à partir de 1.

**Livrable.** `rang_exactitude`, `rang_cout`.
**Critère.** Neuf stratégies, deux ordres. Le premier est celui qu'une compétition affiche, le
second celui qu'une clinique paie.


In [25]:
controle('les deux classements portent les neuf stratégies',
         lambda: len(rang_exactitude) == len(lignes) == len(rang_cout),
         'une entrée par stratégie de lignes')
controle('les mêmes noms dans les deux classements',
         lambda: set(rang_exactitude) == set(rang_cout) == set(lignes['stratégie']),
         'aucune stratégie perdue au tri')
controle('les rangs vont de 1 au nombre de stratégies',
         lambda: sorted(rang_exactitude.values()) == list(range(1, len(lignes) + 1))
                 and sorted(rang_cout.values()) == list(range(1, len(lignes) + 1)),
         'des rangs uniques, sans trou ni doublon')
controle('le premier rang est bien le meilleur de chaque tri',
         lambda: rang_exactitude[max(lignes['stratégie'],
                                     key=lambda s: float(lignes.loc[
                                         lignes['stratégie'] == s, 'exactitude (%)'].iloc[0]))] == 1
                 and rang_cout[min(lignes['stratégie'],
                                   key=lambda s: float(lignes.loc[
                                       lignes['stratégie'] == s, 'coût'].iloc[0]))] == 1,
         'la meilleure exactitude et le plus petit coût ont le rang 1')


  (pas encore écrit)  les deux classements portent les neuf stratégies
  (pas encore écrit)  les mêmes noms dans les deux classements
  (pas encore écrit)  les rangs vont de 1 au nombre de stratégies
  (pas encore écrit)  le premier rang est bien le meilleur de chaque tri


## C2 — les inversions, comptées

**Consigne.** Comptez les **paires** de stratégies qui occupent un ordre différent dans les
deux classements. Deux stratégies forment une inversion quand celle qui passe devant l'autre
par l'exactitude passe derrière par le coût.

**Livrable.** `inversions` — un entier, au plus 36 pour neuf stratégies.
**Critère.** « Les deux classements ne désignent pas la même tête » est une impression
jusqu'à ce qu'elle soit chiffrée.


In [26]:
controle('`inversions` est un compte de paires',
         lambda: isinstance(inversions, (int, np.integer)) and inversions >= 0,
         'un entier, jamais un pourcentage')
controle('le compte ne dépasse pas le nombre de paires',
         lambda: int(inversions) <= len(lignes) * (len(lignes) - 1) // 2,
         'au plus n × (n − 1) / 2 paires')
controle('les deux têtes sont nommées',
         lambda: tete_exactitude in rang_exactitude and tete_cout in rang_cout,
         'une stratégie de lignes chacune')
controle('chaque tête est bien au rang 1',
         lambda: rang_exactitude[tete_exactitude] == 1 and rang_cout[tete_cout] == 1,
         'les deux noms désignent les deux premiers rangs')


  (pas encore écrit)  `inversions` est un compte de paires
  (pas encore écrit)  le compte ne dépasse pas le nombre de paires
  (pas encore écrit)  les deux têtes sont nommées
  (pas encore écrit)  chaque tête est bien au rang 1


## C3 — ce que coûte de choisir la tête affichée

**Consigne.** Chiffrez `cout_du_choix_affiche` : le coût supplémentaire qu'entraîne de déployer
la tête du classement par exactitude au lieu de la tête du classement par coût. Rendez-le
aussi en créneaux équivalents dans `cout_du_choix_creneaux`.

**Livrable.** `cout_du_choix_affiche`, `cout_du_choix_creneaux`.
**Critère.** Un regret non chiffré reste une opinion.


In [27]:
controle('`cout_du_choix_affiche` est la différence des deux stratégies',
         lambda: abs(float(cout_du_choix_affiche) -
                     (float(lignes.loc[lignes['stratégie'] == tete_exactitude, 'coût'].iloc[0])
                      - float(lignes.loc[lignes['stratégie'] == tete_cout, 'coût'].iloc[0]))) < 1,
         'coût de la tête affichée moins coût de la tête réelle')
controle('le regret ne peut pas être négatif',
         lambda: float(cout_du_choix_affiche) >= 0,
         'si le signe est négatif, les deux têtes sont inversées')
controle('le regret en créneaux divise par COUT_FN',
         lambda: abs(float(cout_du_choix_creneaux) -
                     float(cout_du_choix_affiche) / COUT_FN) < 0.01,
         'une absence manquée = COUT_FN unités')


  (pas encore écrit)  `cout_du_choix_affiche` est la différence des deux stratégies
  (pas encore écrit)  le regret ne peut pas être négatif
  (pas encore écrit)  le regret en créneaux divise par COUT_FN


## C4 — la tête tient-elle à un tirage ?

**Consigne.** Refaites le découpage avec une autre graine (`random_state=1`), réentraînez, 
recalculez `lignes`, et retrouvez la tête du classement par exactitude sur ce second tirage.
Comptez combien de têtes différentes apparaissent sur les deux découpages
(`tete_stable`).

**Livrable.** `tete_stable` — un entier, 1 ou 2 — et `tetes_observees` — la liste des têtes
rencontrées.
**Critère.** Une tête qui change avec la graine n'est pas un résultat : c'est un tirage qui a
gagné.


In [28]:
controle('`tetes_observees` porte une ou deux têtes',
         lambda: isinstance(tetes_observees, (list, tuple, set))
                 and 1 <= len(set(tetes_observees)) <= 2,
         'une par découpage : deux au plus')
controle('`tete_stable` compte les têtes distinctes',
         lambda: int(tete_stable) == len(set(tetes_observees)),
         'le nombre de têtes différentes, pas un booléen')
controle('la première tête observée est celle de votre classement',
         lambda: tete_exactitude in set(tetes_observees),
         'le premier découpage est celui de lignes')


  (pas encore écrit)  `tetes_observees` porte une ou deux têtes
  (pas encore écrit)  `tete_stable` compte les têtes distinctes
  (pas encore écrit)  la première tête observée est celle de votre classement


## C5 — la règle en place entre dans le classement

**Contexte.** La séance 3 a posé une règle simple : annoncer une absence si le délai dépasse
`seuil_appris` jours. Elle perd contre `constante` en exactitude. Perd-elle aussi en coût ?

**Consigne.** Ajoutez la règle `regle` comme dixième stratégie. Mesurez son exactitude et son
coût (`regle_en_place`), puis dites si elle bat `constante` en coût (`regle_bat_constante`).

**Livrable.** `regle_en_place` — exactitude et coût — et `regle_bat_constante`.
**Critère.** Une règle simple qu'on déploie déjà n'est pas jugée sur la bonne métrique
seulement : elle se juge sur celle qui décide.


In [29]:
controle('`regle_en_place` porte les deux mesures',
         lambda: set(map(str, regle_en_place)) ==
                 {'exactitude (%)', 'coût'},
         'deux clés : l exactitude et le coût')
controle('l exactitude de la règle est un pourcentage',
         lambda: 0 <= float(regle_en_place['exactitude (%)']) <= 100,
         'entre 0 et 100')
controle('l exactitude est celle que la mesure donne pour regle',
         lambda: abs(float(regle_en_place['exactitude (%)'])
                     - 100 * (regle == yval).mean()) < 0.01,
         'recalculée depuis regle et yval')
controle('le coût est celui que cout() donne pour regle',
         lambda: int(regle_en_place['coût']) == int(cout(regle)),
         'la fonction cout appliquée à regle')
controle('`regle_bat_constante` est un booléen',
         lambda: isinstance(regle_bat_constante, (bool, np.bool_)),
         'vrai ou faux')
controle('le verdict compare les deux coûts',
         lambda: bool(regle_bat_constante) ==
                 (int(cout(regle)) < int(cout(np.zeros(len(yval), dtype=int)))),
         'la règle contre la réponse constante, en coût')


  (pas encore écrit)  `regle_en_place` porte les deux mesures
  (pas encore écrit)  l exactitude de la règle est un pourcentage
  (pas encore écrit)  l exactitude est celle que la mesure donne pour regle
  (pas encore écrit)  le coût est celui que cout() donne pour regle
  (pas encore écrit)  `regle_bat_constante` est un booléen
  (pas encore écrit)  le verdict compare les deux coûts


# Volet D — l'interface

**Contexte.** Le cours appelle une fonction, pas un carnet : `predict(df)` reçoit une table
**brute** et rend un 0/1 par ligne. La chaîne `preparer` est fournie ; l'interface, non.


## D1 — écrire `predict`, et prouver qu'elle est la bonne

**Consigne.** Écrivez `predict(df)`. Elle doit passer par `preparer`, jamais refaire la chaîne
à la main : c'est ce couplage qui garantit qu'une table n'est jamais préparée de deux façons.

Prouvez ensuite qu'elle reproduit le modèle. Prenez les 500 premières lignes brutes de `train`,
appliquez `predict`, et comparez au modèle appliqué à la même table préparée. Le résultat
concordant s'écrit `predict_verifie`.

**Livrable.** votre fonction `predict(df)` et `predict_verifie` — un booléen.
**Critère.** Une interface qui ne passe pas par la chaîne répond sur une autre table que celle
de l'entraînement : elle a l'air juste et elle est fausse.


In [30]:
controle('`predict` accepte une table brute et rend un 0/1 par ligne',
         lambda: len(predict(train.head(50))) == 50
                 and set(np.unique(predict(train.head(50)))) <= {0, 1},
         'cinquante lignes brutes, cinquante réponses 0 ou 1')
controle('`predict` passe par la chaîne fournie',
         lambda: np.array_equal(np.asarray(predict(train.head(50))).ravel(),
                                modele.predict(preparer(train.head(50))).astype(int)),
         'elle doit rendre ce que preparer donne au modèle')
controle('`predict_verifie` est un booléen',
         lambda: isinstance(predict_verifie, (bool, np.bool_)),
         'vrai ou faux')
controle('la preuve porte sur cinq cents lignes',
         lambda: bool(predict_verifie) is True,
         'la comparaison doit concorder, pas seulement exister')


  (pas encore écrit)  `predict` accepte une table brute et rend un 0/1 par ligne
  (pas encore écrit)  `predict` passe par la chaîne fournie
  (pas encore écrit)  `predict_verifie` est un booléen
  (pas encore écrit)  la preuve porte sur cinq cents lignes


# Volet E — le verdict

**Contexte.** S1 à S4 ont appris à mesurer. E1 se termine sur une décision défendable, écrite
par vous.


## E1 — dix lignes, à la première personne

**Consigne.** Rédigez `mon_verdict`, dix lignes qui répondent dans cet ordre :

1. la métrique que **vous** défendez pour ce problème, et ce qu'elle récompense ;
2. le seuil que vous retenez, et pourquoi celui-là plutôt que `0,50` ;
3. ce que coûte de s'être fié à la métrique affichée — en unités de coût et en créneaux ;
4. ce que votre résultat **ne** prouve pas : le découpage, la calibration des probabilités, ou
   la taille de votre échantillon.

**Livrable.** `mon_verdict` — une chaîne de dix lignes.
**Critère.** Un verdict qui ne nomme aucune limite est incomplet. Ce que vous ne savez pas
encore se dit en une ligne, et cette ligne est notée.


In [31]:
controle('`mon_verdict` fait dix lignes',
         lambda: len([l for l in str(mon_verdict).splitlines() if l.strip()]) == 10,
         'dix lignes, pas neuf ni onze')
controle('`mon_verdict` est écrit à la première personne',
         lambda: any(m in str(mon_verdict).lower() for m in
                     ('je ', 'j’', "j'", 'mon ', 'ma ')),
         'je, mon, ma : c est votre décision')
controle('le verdict nomme une limite',
         lambda: any(m in str(mon_verdict).lower() for m in
                     ('ne prouve', 'limite', 'calibration', 'découpage', 'échantillon', 'bruit')),
         'ce que le résultat ne prouve pas')
controle('le verdict cite la facture',
         lambda: str(int(round(float(facture_de_la_confiance)))) in str(mon_verdict).replace(' ', '')
                 or f"{float(facture_creneaux):.0f}" in str(mon_verdict),
         'le coût de s être fié à la métrique affichée, en unités ou en créneaux')


  (pas encore écrit)  `mon_verdict` fait dix lignes
  (pas encore écrit)  `mon_verdict` est écrit à la première personne
  (pas encore écrit)  le verdict nomme une limite
  (pas encore écrit)  le verdict cite la facture


# Rendre le travail

Cette cellule écrit le relevé que vous déposez avec le carnet. Elle porte une ligne par
contrôle : son nom, et s'il tient.


In [32]:
# le relevé — une ligne par contrôle, à pousser dans votre dépôt avec le carnet
bilan()
releve = pd.DataFrame(VERIFICATIONS, columns=['contrôle', 'tient'])
releve.to_csv('verifications.csv', index=False)
print(f'{len(releve)} contrôles écrits dans verifications.csv')
display(releve)


vos contrôles : 0 ok sur 0
0 contrôles écrits dans verifications.csv


,contrôle,tient


## Ce que vous déposez, à la clôture de la séance

- le **relevé** `verifications.csv`, écrit par la cellule ci-dessus ;
- ce **carnet**, exécuté en `Save & Run All` (privé), avec la graine fixée : deux exécutions
  doivent rendre le même chiffre ;
- le **module `.py`** qui expose `predict(df)` — la fonction du volet D, sortie du carnet ;
- `declaration_E1.md`, copié du modèle publié au dépôt du cours (`DECLARATION_modele.md`).

Les quatre vont dans le **même commit**. Sans la déclaration, le rendu est incomplet.

**Barème — annoncé d'avance, 100 points.**

| Critère | Points | Ce qu'il regarde |
|---|---|---|
| Justesse de la mesure | **40** | vos chiffres, et leur cohérence avec le relevé |
| Interprétation et verdict | **30** | ce que vous concluez, et ce que vous refusez de conclure |
| Reproductibilité du livrable | **20** | carnet réexécutable, graine fixée, même score |
| Lisibilité du journal | **10** | le `mon_verdict`, et les raisons écrites |

Les contrôles ne donnent jamais la réponse. Chaque question porte un contrôle de **forme** :
il vérifie un type, une plage, une dimension, ou la cohérence avec vos autres réponses. Tant
que la réponse n'est pas écrite, il se tait ; écrit, il vous dit s'il tient ou s'il est à
revoir.
